In [6]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
from optuna.samplers import CmaEsSampler

warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────────────
DATA_DIR        = os.environ.get("DATA_DIR", ".")
TRAIN_PATH      = os.path.join(DATA_DIR, "train.csv")
TEST_PATH       = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = "submission.csv"

TARGET   = "demand"
ID_COL   = "Index"
N_SPLITS = 10                  # Back to 10 folds for max accuracy
SEED     = 42
MULTI_SEEDS = [42, 123, 2024]  # Back to 3-seed averaging for stability
print("Imports OK")

Imports OK


In [7]:
# ── Geohash utilities ──────────────────────────────────────────────────────────────
_BASE32    = "0123456789bcdefghjkmnpqrstuvwxyz"
_DECODE_MAP = {c: i for i, c in enumerate(_BASE32)}

# FIXED: Corrected neighbour direction offsets (removed corrupted strings)
_GH_NEIGHBOUR = {
    'n': {'even': 'p0r21436x8zb9dcf5h7kjnmqesgutwvy', 'odd': 'bc01fg45238967deuvhjyznpkmstqrwx'},
    's': {'even': '14365h7k9dcfesgujnmqp0r2twvyx8zb', 'odd': '238967debc01fg45uvhjyznpkmstqrwx'},
    'e': {'even': 'bc01fg45238967deuvhjyznpkmstqrwx', 'odd': 'p0r21436x8zb9dcf5h7kjnmqesgutwvy'},
    'w': {'even': '238967debc01fg45uvhjyznpkmstqrwx', 'odd': '14365h7k9dcfesgujnmqp0r2twvyx8zb'},
}

_GH_BORDER = {
    'n': {'even': 'prxz', 'odd': 'bcfguvyz'},
    's': {'even': '028b', 'odd': '0145hjnp'},
    'e': {'even': 'bcfguvyz', 'odd': 'prxz'},
    'w': {'even': '0145hjnp', 'odd': '028b'},
}


def _decode_geohash(geohash_str):
    """Decode a geohash string into (latitude, longitude) centre point."""
    if not isinstance(geohash_str, str) or geohash_str == "":
        return np.nan, np.nan
    lat_lo, lat_hi = -90.0,  90.0
    lon_lo, lon_hi = -180.0, 180.0
    is_even = True
    for ch in geohash_str.lower():
        cd = _DECODE_MAP.get(ch)
        if cd is None:
            continue
        for mask in (16, 8, 4, 2, 1):
            if is_even:
                mid = (lon_lo + lon_hi) / 2.0
                if cd & mask: lon_lo = mid
                else:         lon_hi = mid
            else:
                mid = (lat_lo + lat_hi) / 2.0
                if cd & mask: lat_lo = mid
                else:         lat_hi = mid
            is_even = not is_even
    return (lat_lo + lat_hi) / 2.0, (lon_lo + lon_hi) / 2.0


def geohash_neighbour(geohash_str, direction):
    """Compute the adjacent geohash in a given direction ('n','s','e','w')."""
    if not geohash_str or not isinstance(geohash_str, str):
        return None
    gh = geohash_str.lower()
    last_char = gh[-1]
    parent = gh[:-1]
    parity = 'even' if len(gh) % 2 == 1 else 'odd'

    if last_char in _GH_BORDER[direction][parity] and parent:
        parent = geohash_neighbour(parent, direction)
        if parent is None:
            return None

    idx = _GH_NEIGHBOUR[direction][parity].index(last_char)
    return parent + _BASE32[idx]

print("Geohash utils OK")

Geohash utils OK


In [8]:
# ── Day parsing ────────────────────────────────────────────────────────────────────
def parse_day_column(series):
    day_num = pd.to_numeric(series, errors="coerce")
    if day_num.notna().any():
        return (day_num % 7).astype(float)
    day_dt = pd.to_datetime(series, errors="coerce")
    if day_dt.notna().any():
        return day_dt.dt.dayofweek.astype(float)
    name_map = {
        "monday": 0, "tuesday": 1, "wednesday": 2, "thursday": 3,
        "friday": 4, "saturday": 5, "sunday": 6,
    }
    return series.astype(str).str.strip().str.lower().map(name_map).astype(float)

print("Day parsing OK")

Day parsing OK


In [9]:
# ── Base feature engineering ───────────────────────────────────────────────────────
def build_base_features(train_raw, test_raw):
    train = train_raw.copy()
    test  = test_raw.copy()

    for df in (train, test):

        # ── Spatial: decode geohash ──────────────────────────────────────────
        if "geohash" in df.columns:
            gh = df["geohash"].astype(str)
            cache = {g: _decode_geohash(g) for g in gh.unique()}
            df["gh_lat"]          = gh.map(lambda g: cache[g][0])
            df["gh_lon"]          = gh.map(lambda g: cache[g][1])
            df["geohash_prefix_4"] = gh.str[:4]
            df["geohash_prefix_5"] = gh.str[:5]
            df["geohash_prefix_3"] = gh.str[:3]
            df["geohash_len"]      = gh.str.len().astype(int)

        # ── Intelligent imputation: RoadType (geohash-mode → global-mode) ───
        if "RoadType" in df.columns:
            _rt_global = df["RoadType"].mode()
            _rt_global = _rt_global.iloc[0] if len(_rt_global) > 0 else "Unknown"
            if "geohash" in df.columns:
                _rt_gh_mode = df.groupby("geohash")["RoadType"].transform(
                    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
                )
                df["RoadType"] = df["RoadType"].fillna(_rt_gh_mode)
            df["RoadType"] = df["RoadType"].fillna(_rt_global)

        # ── Temporal: parse timestamp ────────────────────────────────────────
        if "timestamp" in df.columns:
            ts = df["timestamp"]
            dt = pd.to_datetime(ts, errors="coerce")
            if dt.isna().all():
                num = pd.to_numeric(ts, errors="coerce")
                if num.notna().any():
                    unit = "ms" if num.dropna().median() > 1e11 else "s"
                    dt   = pd.to_datetime(num, unit=unit, errors="coerce")
            if dt.isna().all():
                parts = ts.astype(str).str.split(":", expand=True)
                if parts.shape[1] >= 2:
                    hours   = pd.to_numeric(parts[0], errors="coerce").fillna(0)
                    minutes = pd.to_numeric(parts[1], errors="coerce").fillna(0)
                    df["ts_hour"]      = hours.astype(int)
                    df["ts_minute"]    = minutes.astype(int)
                    df["ts_dayofweek"] = np.nan
                    df["ts_day"]       = np.nan
                    df["ts_month"]     = np.nan
                else:
                    for c in ["ts_hour","ts_minute","ts_dayofweek","ts_day","ts_month"]:
                        df[c] = np.nan
            else:
                df["ts_hour"]      = dt.dt.hour
                df["ts_minute"]    = dt.dt.minute
                df["ts_dayofweek"] = dt.dt.dayofweek
                df["ts_day"]       = dt.dt.day
                df["ts_month"]     = dt.dt.month

        # ── Unified 96-slot time index & cyclical encoding ───────────────
        hod = df["ts_hour"].fillna(0).astype(float)
        mod = df["ts_minute"].fillna(0).astype(float)
        df["time_slot"] = (hod * 4 + mod // 15).astype(int)
        df["slot_sin"]  = np.sin(2 * np.pi * df["time_slot"] / 96.0)
        df["slot_cos"]  = np.cos(2 * np.pi * df["time_slot"] / 96.0)

        # ── Intelligent imputation: Temperature (time_slot mean → global) ──
        if "Temperature" in df.columns:
            df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce")
            _slot_temp_mean = df.groupby("time_slot")["Temperature"].transform("mean")
            df["Temperature"] = df["Temperature"].fillna(_slot_temp_mean)
            df["Temperature"] = df["Temperature"].fillna(df["Temperature"].mean())

        if "day" in df.columns:
            df["day_num"] = parse_day_column(df["day"])
            if df["ts_dayofweek"].isna().all():
                df["ts_dayofweek"] = df["day_num"]
        else:
            df["day_num"] = df.get("ts_dayofweek", pd.Series(np.nan, index=df.index))

        dow = df["ts_dayofweek"].fillna(0).astype(float)
        df["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
        df["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)

        # ── Finer time-of-day bins ───────────────────────────────────────────
        hour_vals = df["ts_hour"].fillna(-1).astype(int)
        bins  = [-1,5,8,11,14,17,20,23]
        df["time_segment"] = pd.cut(hour_vals, bins=bins, labels=False).fillna(0).astype(int)

        # ── Binary flags ─────────────────────────────────────────────────────
        df["is_weekend"]  = (df["ts_dayofweek"].fillna(0) >= 5).astype(int)
        df["rush_hour"]   = hour_vals.isin({7, 8, 9, 17, 18, 19}).astype(int)
        df["night_flag"]  = ((hour_vals >= 22) | (hour_vals <= 5)).astype(int)
        df["is_monday"]   = (df["ts_dayofweek"].fillna(-1) == 0).astype(int)
        df["is_friday"]   = (df["ts_dayofweek"].fillna(-1) == 4).astype(int)
        df["is_workday_rush"] = (
            (df["is_weekend"] == 0) & (df["rush_hour"] == 1)
        ).astype(int)

        # ── Binary / numeric cleanup ─────────────────────────────────────────
        if "LargeVehicles" in df.columns:
            lv = df["LargeVehicles"].astype(str).str.strip().str.lower()
            df["LargeVehicles"] = lv.map(
                lambda x: 1 if x in ("1", "yes", "true", "allowed") else 0
            ).astype(int)
        if "Landmarks" in df.columns:
            lm = df["Landmarks"].astype(str).str.strip().str.lower()
            df["Landmarks"] = lm.map(
                lambda x: 1 if x in ("1", "yes", "true") else 0
            ).astype(int)

        # ── Interaction features ─────────────────────────────────────────────
        if "NumberofLanes" in df.columns and "Temperature" in df.columns:
            lanes = pd.to_numeric(df["NumberofLanes"], errors="coerce").fillna(0)
            temp  = pd.to_numeric(df["Temperature"],   errors="coerce").fillna(0)
            df["lanes_x_temp"]  = lanes * temp
            df["lanes_squared"] = lanes ** 2
            df["lanes_x_rush"]  = lanes * df["rush_hour"]
            df["lanes_x_wknd"]  = lanes * df["is_weekend"]
            df["temp_squared"]  = temp ** 2

        if "gh_lat" in df.columns:
            df["lat_x_slotsin"] = df["gh_lat"] * df["slot_sin"]
            df["lon_x_slotcos"] = df["gh_lon"] * df["slot_cos"]
            df["lat_x_dow"]     = df["gh_lat"] * dow
            df["lon_x_dow"]     = df["gh_lon"] * dow

        df["hour_x_dow"] = hod * dow

    # ── Spatial Clustering (K-Means) ─────────────────────────────────────────
    if "gh_lat" in train.columns and "gh_lon" in train.columns:
        coords_train = train[["gh_lat", "gh_lon"]].dropna()
        if not coords_train.empty:
            kmeans = KMeans(n_clusters=50, random_state=SEED, n_init=10)
            train.loc[coords_train.index, "spatial_cluster"] = kmeans.fit_predict(coords_train)

            coords_test = test[["gh_lat", "gh_lon"]].dropna()
            if not coords_test.empty:
                test.loc[coords_test.index, "spatial_cluster"] = kmeans.predict(coords_test)

            train["cluster_x_hour"] = train["spatial_cluster"].astype(str) + "_" + train["ts_hour"].astype(str)
            test["cluster_x_hour"] = test["spatial_cluster"].astype(str) + "_" + test["ts_hour"].astype(str)

    # ── Geohash frequency encoding ──────────────────────────────────────────
    if "geohash" in train.columns:
        gh_counts  = train["geohash"].value_counts().to_dict()
        p3_counts  = train["geohash_prefix_3"].value_counts().to_dict()
        train["gh_freq"]   = train["geohash"].map(gh_counts).fillna(0).astype(int)
        test["gh_freq"]    = test["geohash"].map(gh_counts).fillna(0).astype(int)
        train["gh3_freq"]  = train["geohash_prefix_3"].map(p3_counts).fillna(0).astype(int)
        test["gh3_freq"]   = test["geohash_prefix_3"].map(p3_counts).fillna(0).astype(int)
        rank_map = {k: r for r, k in enumerate(
            sorted(gh_counts, key=gh_counts.get, reverse=True), 1
        )}
        train["gh_rank"] = train["geohash"].map(rank_map).fillna(len(rank_map)+1).astype(int)
        test["gh_rank"]  = test["geohash"].map(rank_map).fillna(len(rank_map)+1).astype(int)

    # ── Temperature binning ────────────────────────────────────────────────
    if "Temperature" in train.columns:
        train_temp = pd.to_numeric(train["Temperature"], errors="coerce")
        test_temp  = pd.to_numeric(test["Temperature"],  errors="coerce")
        train["Temperature"] = train_temp
        test["Temperature"]  = test_temp
        try:
            _, bin_edges = pd.qcut(train_temp.dropna(), q=10,
                                   retbins=True, duplicates="drop")
            train["temp_bin"] = pd.cut(train_temp, bins=bin_edges,
                                       labels=False, include_lowest=True)
            test["temp_bin"]  = pd.cut(test_temp,  bins=bin_edges,
                                       labels=False, include_lowest=True)
        except ValueError:
            train["temp_bin"] = 0
            test["temp_bin"]  = 0

    if "NumberofLanes" in train.columns:
        train["NumberofLanes"] = pd.to_numeric(train["NumberofLanes"], errors="coerce")
        test["NumberofLanes"]  = pd.to_numeric(test["NumberofLanes"],  errors="coerce")

    return train, test

print("Feature engineering function defined")

Feature engineering function defined


In [10]:
# ── Target encoding config ───────────────────────────────────────────────────────
TARGET_ENCODE_KEYS = [
    (["geohash"],                          "te_geohash"),
    (["RoadType"],                         "te_RoadType"),
    (["Weather"],                          "te_Weather"),
    (["geohash_prefix_3"],                 "te_geohash_prefix_3"),
    (["geohash_prefix_4"],                 "te_geohash_prefix_4"),
    (["geohash_prefix_5"],                 "te_geohash_prefix_5"),
    (["spatial_cluster"],                  "te_spatial_cluster"),
    (["spatial_cluster", "ts_hour"],       "te_cluster_x_hour"),
    (["geohash", "ts_hour"],               "agg_gh_hour"),
    (["geohash", "day_num"],               "agg_gh_day"),
    (["geohash", "time_segment"],          "agg_gh_segment"),
    (["RoadType", "ts_hour"],              "agg_rt_hour"),
    (["RoadType", "is_weekend"],           "agg_rt_wknd"),
    (["ts_hour",  "is_weekend"],           "agg_hour_wknd"),
    (["Weather",  "RoadType"],             "agg_weather_rt"),
    (["Weather",  "ts_hour"],              "agg_weather_hour"),
    (["geohash", "is_weekend"],            "agg_gh_wknd"),
    (["geohash_prefix_4", "ts_hour"],      "agg_gh4_hour"),
    # NEW: 3-way interaction and time_slot grouping
    (["geohash", "ts_hour", "day_num"],    "agg_gh_hour_day"),
    (["geohash", "time_slot"],             "agg_gh_slot"),
]

SMOOTHING = 10


def _make_group_key(df, cols):
    if len(cols) == 1:
        return df[cols[0]].astype(str)
    return df[cols].astype(str).apply("_".join, axis=1)


def target_encode_oof(train, test, y, all_folds):
    """
    OOF smoothed target encoding: computes mean, std, median per group.
    """
    global_mean_full   = y.mean()
    global_std_full    = y.std()
    global_median_full = y.median()

    for group_cols, feat_name in TARGET_ENCODE_KEYS:
        missing = [c for c in group_cols if c not in train.columns]
        if missing:
            print(f"  [TE] Skipping {feat_name}: missing columns {missing}")
            train[feat_name]            = global_mean_full
            test[feat_name]             = global_mean_full
            train[feat_name + "_std"]   = global_std_full
            test[feat_name + "_std"]    = global_std_full
            train[feat_name + "_med"]   = global_median_full
            test[feat_name + "_med"]    = global_median_full
            continue

        train_key = _make_group_key(train, group_cols)
        test_key  = _make_group_key(test,  group_cols)

        enc_mean   = np.full(len(train), np.nan, dtype=np.float64)
        enc_std    = np.full(len(train), np.nan, dtype=np.float64)
        enc_median = np.full(len(train), np.nan, dtype=np.float64)

        # ---- OOF encoding (vectorized per fold) ---
        for tr_idx, va_idx in all_folds:
            fold_y = y.iloc[tr_idx]
            fold_global_mean   = fold_y.mean()
            fold_global_std    = fold_y.std()
            fold_global_median = fold_y.median()

            tmp = pd.DataFrame({"key": train_key.iloc[tr_idx], "y": fold_y})
            stats = tmp.groupby("key")["y"].agg(["sum", "count", "std", "median"])
            stats["std"] = stats["std"].fillna(0)

            va_keys = train_key.iloc[va_idx]
            va_cnt  = va_keys.map(stats["count"]).fillna(0)

            # Mean
            va_sum  = va_keys.map(stats["sum"]).fillna(0)
            fold_mean = (va_sum / va_cnt.replace(0, np.nan)).fillna(fold_global_mean)
            enc_mean[va_idx] = (
                (va_cnt * fold_mean + fold_global_mean * SMOOTHING) / (va_cnt + SMOOTHING)
            ).values

            # Std
            va_std = va_keys.map(stats["std"]).fillna(0)
            enc_std[va_idx] = (
                (va_cnt * va_std + fold_global_std * SMOOTHING) / (va_cnt + SMOOTHING)
            ).values

            # Median
            va_med = va_keys.map(stats["median"]).fillna(fold_global_median)
            enc_median[va_idx] = (
                (va_cnt * va_med + fold_global_median * SMOOTHING) / (va_cnt + SMOOTHING)
            ).values

        train[feat_name]          = enc_mean
        train[feat_name + "_std"] = enc_std
        train[feat_name + "_med"] = enc_median

        # ---- Full-data encoding for test ---
        tmp_full = pd.DataFrame({"key": train_key, "y": y})
        full_stats = tmp_full.groupby("key")["y"].agg(["sum", "count", "std", "median"])
        full_stats["std"] = full_stats["std"].fillna(0)

        te_cnt = test_key.map(full_stats["count"]).fillna(0)

        # Mean
        te_sum  = test_key.map(full_stats["sum"]).fillna(0)
        te_mean = (te_sum / te_cnt.replace(0, np.nan)).fillna(global_mean_full)
        test[feat_name] = (
            (te_cnt * te_mean + global_mean_full * SMOOTHING) / (te_cnt + SMOOTHING)
        ).values

        # Std
        te_std = test_key.map(full_stats["std"]).fillna(0)
        test[feat_name + "_std"] = (
            (te_cnt * te_std + global_std_full * SMOOTHING) / (te_cnt + SMOOTHING)
        ).values

        # Median
        te_med = test_key.map(full_stats["median"]).fillna(global_median_full)
        test[feat_name + "_med"] = (
            (te_cnt * te_med + global_median_full * SMOOTHING) / (te_cnt + SMOOTHING)
        ).values

        # NaN fill
        train[feat_name]          = train[feat_name].fillna(global_mean_full)
        test[feat_name]           = test[feat_name].fillna(global_mean_full)
        train[feat_name + "_std"] = train[feat_name + "_std"].fillna(global_std_full)
        test[feat_name + "_std"]  = test[feat_name + "_std"].fillna(global_std_full)
        train[feat_name + "_med"] = train[feat_name + "_med"].fillna(global_median_full)
        test[feat_name + "_med"]  = test[feat_name + "_med"].fillna(global_median_full)

    return train, test

print("Target encoding function defined")

Target encoding function defined


In [11]:
# ── PIPELINE: Load Data → Features → Lag → KFold → TE → Neighbour TE → Matrix ─

print("Loading data ...")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

# Raw counts – Poisson objectives handle the distribution
y = train_raw[TARGET].astype(float)

print("Building base features ...")
train, test = build_base_features(train_raw, test_raw)

# ── Day-48 Demand Lag (Phase 3) ────────────────────────────────────────
print("Building demand lag features ...")
if "geohash" in train.columns and "timestamp" in train.columns:
    # Build lookup from Day 48 training data: (geohash_timestamp) -> demand
    _lag_key_tr = train["geohash"].astype(str) + "_" + train["timestamp"].astype(str)
    _lag_lookup = dict(zip(_lag_key_tr, y))
    _gh_mean_demand = y.groupby(train["geohash"]).mean().to_dict()
    _global_mean_demand = y.mean()

    # Train: no prior day available → use geohash mean as proxy
    train["demand_lag1"] = train["geohash"].map(_gh_mean_demand).fillna(_global_mean_demand)

    # Test (Day 49): exact match from Day 48, fallback to geohash mean
    _lag_key_te = test["geohash"].astype(str) + "_" + test["timestamp"].astype(str)
    test["demand_lag1"] = _lag_key_te.map(_lag_lookup)
    test["demand_lag1"] = test["demand_lag1"].fillna(
        test["geohash"].map(_gh_mean_demand)
    ).fillna(_global_mean_demand)
    print(f"  demand_lag1: train fill rate = {train['demand_lag1'].notna().mean():.3f}, "
          f"test exact-match rate = {_lag_key_te.map(_lag_lookup).notna().mean():.3f}")

# ── KFold (10 folds, shuffled) ────────────────────────────────────────
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
all_folds = list(kf.split(train, y))
print(f"  KFold: {N_SPLITS} folds materialised")

# ── OOF Target Encoding (mean + std + median) ───────────────────────
print("Applying OOF target encoding (mean/std/median) ...")
train, test = target_encode_oof(train, test, y, all_folds)
print("  Done.")

# ── Geohash Neighbour Average (Phase 3) ───────────────────────────────
print("Computing geohash neighbour TE mean ...")
if "geohash" in train.columns and "te_geohash" in train.columns:
    # Build geohash → te_geohash map from combined train+test
    _all_te = pd.concat([
        train[["geohash", "te_geohash"]],
        test[["geohash", "te_geohash"]]
    ])
    _gh_te_map = _all_te.groupby("geohash")["te_geohash"].mean().to_dict()
    _global_te = np.mean(list(_gh_te_map.values()))

    # Compute neighbour TE for each unique geohash (vectorized lookup)
    _unique_ghs = set(train["geohash"].unique()) | set(test["geohash"].unique())
    _gh_nbr_te = {}
    for _gh in _unique_ghs:
        _te_vals = []
        for _d in ['n', 's', 'e', 'w']:
            _nbr = geohash_neighbour(_gh, _d)
            if _nbr and _nbr in _gh_te_map:
                _te_vals.append(_gh_te_map[_nbr])
        _gh_nbr_te[_gh] = np.mean(_te_vals) if _te_vals else _global_te

    train["neighbor_te_mean"] = train["geohash"].map(_gh_nbr_te).fillna(_global_te)
    test["neighbor_te_mean"]  = test["geohash"].map(_gh_nbr_te).fillna(_global_te)
    print(f"  neighbor_te_mean computed for {len(_gh_nbr_te)} unique geohashes")

# ── Prepare feature matrix ────────────────────────────────────────────
drop_cols = {
    ID_COL, TARGET, "timestamp", "day", "geohash",
    "geohash_prefix_3", "geohash_prefix_4", "geohash_prefix_5",
    "RoadType", "Weather", "cluster_x_hour",
}
features = [c for c in train.columns if c not in drop_cols and c in test.columns]

for col in features:
    train[col] = pd.to_numeric(train[col], errors="coerce")
    test[col]  = pd.to_numeric(test[col],  errors="coerce")

for col in features:
    col_mean   = train[col].mean()
    train[col] = train[col].fillna(col_mean)
    test[col]  = test[col].fillna(col_mean)

X_train = train[features]
X_test  = test[features]
print(f"  {len(features)} features ready")

Loading data ...
Building base features ...
Building demand lag features ...
  demand_lag1: train fill rate = 1.000, test exact-match rate = 0.889
  KFold: 10 folds materialised
Applying OOF target encoding (mean/std/median) ...
  Done.
Computing geohash neighbour TE mean ...
  neighbor_te_mean computed for 1259 unique geohashes
  102 features ready


In [12]:
import numpy as np
import xgboost as xgb
from catboost import CatBoostRegressor

print("Checking GPU compatibility with actual variance...")
try:
    xgb.XGBRegressor(device='cuda').fit(np.random.rand(10, 2), np.random.rand(10))
    print("✅ XGBoost GPU is ready.")
except Exception as e:
    print("❌ XGBoost GPU error:", e)

try:
    CatBoostRegressor(task_type='GPU', iterations=2).fit(np.random.rand(10, 2), np.random.rand(10), verbose=0)
    print("✅ CatBoost GPU is ready.")
except Exception as e:
    print("❌ CatBoost GPU error:", e)

Checking GPU compatibility with actual variance...
✅ XGBoost GPU is ready.
✅ CatBoost GPU is ready.


In [24]:
# ── FAST OPTUNA HYPERPARAMETER TUNING ──────────────────────────────────────
# 1 fold | 300 estimators | 25 trials | no DB storage
# Expected time: ~8-12 mins total 

import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import r2_score
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

tuning_folds = all_folds 

N_TRIALS     = 25
N_EST        = 300
EARLY_STOP   = 50  # Gives low learning rates (0.02) room to converge


def objective_xgb(trial):
    params = {
        'objective':             'count:poisson',
        'eval_metric':           'rmse',
        'device':                'cuda',
        'tree_method':           'hist',
        'n_estimators':          N_EST,
        'early_stopping_rounds': EARLY_STOP,
        'learning_rate':         trial.suggest_float('learning_rate',    0.02, 0.15, log=True),
        'max_depth':             trial.suggest_int  ('max_depth',        4, 8),
        'subsample':             trial.suggest_float('subsample',        0.6, 1.0),
        'colsample_bytree':      trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda':            trial.suggest_float('reg_lambda',       1e-2, 10.0, log=True),
        'reg_alpha':             trial.suggest_float('reg_alpha',        1e-2, 5.0,  log=True),
        'min_child_weight':      trial.suggest_int  ('min_child_weight', 1, 7),
        'random_state':          SEED,
        'n_jobs':                -1,
    }
    scores = []
    for tr_idx, va_idx in tuning_folds:
        X_tr, y_tr = X_train.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx],  y.iloc[va_idx]
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        scores.append(r2_score(y_va, model.predict(X_va)))
    return float(np.mean(scores))


def objective_cat(trial):
    params = {
        'loss_function':         'Poisson',
        'task_type':             'GPU',
        'iterations':            N_EST,
        'learning_rate':         trial.suggest_float('learning_rate',       0.02, 0.15, log=True),
        'depth':                 trial.suggest_int  ('depth',               4, 8),
        'l2_leaf_reg':           trial.suggest_float('l2_leaf_reg',         1.0, 10.0, log=True),
        'bagging_temperature':   trial.suggest_float('bagging_temperature', 0.0, 5.0),
        'random_strength':       trial.suggest_float('random_strength',     0.0, 5.0),
        'random_seed':           SEED,
        'verbose':               0,
        'allow_writing_files':   False,
    }
    scores = []
    for tr_idx, va_idx in tuning_folds:
        X_tr, y_tr = X_train.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx],  y.iloc[va_idx]
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va),
                  early_stopping_rounds=EARLY_STOP, verbose=0)
        scores.append(r2_score(y_va, model.predict(X_va)))
    return float(np.mean(scores))


def objective_lgb(trial):
    params = {
        'objective':         'poisson',
        'device':            'cpu',
        'n_estimators':      N_EST,
        'learning_rate':     trial.suggest_float('learning_rate',     0.02, 0.15, log=True),
        'num_leaves':        trial.suggest_int  ('num_leaves',        31, 127),
        'subsample':         trial.suggest_float('subsample',         0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree',  0.6, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda',        1e-2, 10.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha',         1e-2, 5.0,  log=True),
        'min_child_samples': trial.suggest_int  ('min_child_samples', 10, 50),
        'min_split_gain':    trial.suggest_float('min_split_gain',    0.0, 0.5),
        'max_bin':           trial.suggest_int  ('max_bin',           63, 255),
        'bagging_freq':      trial.suggest_int  ('bagging_freq',      1, 7),
        'random_state':      SEED,
        'n_jobs':            -1,
        'verbosity':         -1,
    }
    scores = []
    for tr_idx, va_idx in tuning_folds:
        X_tr, y_tr = X_train.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx],  y.iloc[va_idx]
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                  callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                             lgb.log_evaluation(-1)])
        scores.append(r2_score(y_va, model.predict(X_va)))
    return float(np.mean(scores))


def objective_hist(trial):
    params = {
        'loss':                'poisson',
        'max_iter':            N_EST,
        'learning_rate':       trial.suggest_float('learning_rate',    0.02, 0.15, log=True),
        'max_leaf_nodes':      trial.suggest_int  ('max_leaf_nodes',   31, 127),
        'min_samples_leaf':    trial.suggest_int  ('min_samples_leaf', 10, 50),
        'l2_regularization':   trial.suggest_float('l2_regularization',1e-2, 10.0, log=True),
        'max_bins':            trial.suggest_int  ('max_bins',         63, 255),
        'random_state':        SEED,
        'early_stopping':      True,
        'validation_fraction': 0.1,
        'n_iter_no_change':    EARLY_STOP,
    }
    scores = []
    for tr_idx, va_idx in tuning_folds:
        X_tr, y_tr = X_train.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx],  y.iloc[va_idx]
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_va, model.predict(X_va)))
    return float(np.mean(scores))


# ── RUN ─────────────────────────────────────────────────────────────────────
print(f"Fast Optuna | 10 fold | {N_EST} estimators | {N_TRIALS} trials per model\n")

print("[1/4] XGBoost (GPU)...")
study_xgb = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"  Best R²: {study_xgb.best_value:.5f}\n")

print("[2/4] CatBoost (GPU)...")
study_cat = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=True)
print(f"  Best R²: {study_cat.best_value:.5f}\n")

print("[3/4] LightGBM (CPU)...")
study_lgb = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"  Best R²: {study_lgb.best_value:.5f}\n")

print("[4/4] HistGBR (CPU)...")
study_hist = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study_hist.optimize(objective_hist, n_trials=N_TRIALS, show_progress_bar=True)
print(f"  Best R²: {study_hist.best_value:.5f}\n")

best_xgb_params  = study_xgb.best_params
best_cat_params  = study_cat.best_params
best_lgb_params  = study_lgb.best_params
best_hist_params = study_hist.best_params

print("=" * 55)
print("TUNING COMPLETE")
print("=" * 55)
print(f"best_xgb_params  = {best_xgb_params}")
print(f"best_cat_params  = {best_cat_params}")
print(f"best_lgb_params  = {best_lgb_params}")
print(f"best_hist_params = {best_hist_params}")

Fast Optuna | 1 fold | 300 estimators | 25 trials per model

[1/4] XGBoost (GPU)...


  0%|          | 0/25 [00:00<?, ?it/s]

  Best R²: 0.93304

[2/4] CatBoost (GPU)...


  0%|          | 0/25 [00:00<?, ?it/s]

  Best R²: 0.94064

[3/4] LightGBM (CPU)...


  0%|          | 0/25 [00:00<?, ?it/s]

  Best R²: 0.94183

[4/4] HistGBR (CPU)...


  0%|          | 0/25 [00:00<?, ?it/s]

  Best R²: 0.94288

TUNING COMPLETE
best_xgb_params  = {'learning_rate': 0.036231626460504394, 'max_depth': 7, 'subsample': 0.7788723052280968, 'colsample_bytree': 0.9226315528614042, 'reg_lambda': 0.10071174503562488, 'reg_alpha': 0.03672239540543744, 'min_child_weight': 3}
best_cat_params  = {'learning_rate': 0.14087436771639622, 'depth': 7, 'l2_leaf_reg': 9.975452672771397, 'bagging_temperature': 0.02157765421122236, 'random_strength': 3.9050992261570387}
best_lgb_params  = {'learning_rate': 0.05789656828137544, 'num_leaves': 80, 'subsample': 0.8480595631138229, 'colsample_bytree': 0.9173995865175877, 'reg_lambda': 0.5726703217852122, 'reg_alpha': 0.05336621674627308, 'min_child_samples': 33, 'min_split_gain': 0.002856610734086394, 'max_bin': 147, 'bagging_freq': 6}
best_hist_params = {'learning_rate': 0.03583783709038413, 'max_leaf_nodes': 115, 'min_samples_leaf': 45, 'l2_regularization': 0.07988585071126157, 'max_bins': 231}


In [25]:
# ── AUTOMATED HYPERPARAMETERS (OPTUNA OUTPUTS) ─────────────────────────────
print("🔥 OVERNIGHT OPTUNA RUN COMPLETE 🔥")
print("Mapping optimal Optuna parameters directly to final models...\n")

BEST_XGB_PARAMS = {'learning_rate': 0.036231626460504394, 'max_depth': 7, 'subsample': 0.7788723052280968, 'colsample_bytree': 0.9226315528614042, 'reg_lambda': 0.10071174503562488, 'reg_alpha': 0.03672239540543744, 'min_child_weight': 3}
BEST_CAT_PARAMS = {'learning_rate': 0.14087436771639622, 'depth': 7, 'l2_leaf_reg': 9.975452672771397, 'bagging_temperature': 0.02157765421122236, 'random_strength': 3.9050992261570387}
BEST_LGB_PARAMS = {'learning_rate': 0.05789656828137544, 'num_leaves': 80, 'subsample': 0.8480595631138229, 'colsample_bytree': 0.9173995865175877, 'reg_lambda': 0.5726703217852122, 'reg_alpha': 0.05336621674627308, 'min_child_samples': 33, 'min_split_gain': 0.002856610734086394, 'max_bin': 147, 'bagging_freq': 6}
BEST_HIST_PARAMS = {'learning_rate': 0.03583783709038413, 'max_leaf_nodes': 115, 'min_samples_leaf': 45, 'l2_regularization': 0.07988585071126157, 'max_bins': 231}

print("--- LightGBM ---")
for k, v in BEST_LGB_PARAMS.items(): print(f"  {k}: {v}")
print("\n--- XGBoost ---")
for k, v in BEST_XGB_PARAMS.items(): print(f"  {k}: {v}")
print("\n--- CatBoost ---")
for k, v in BEST_CAT_PARAMS.items(): print(f"  {k}: {v}")
print("\n--- HistGBR ---")
for k, v in BEST_HIST_PARAMS.items(): print(f"  {k}: {v}")
print("--------------------------------------------------")

🔥 OVERNIGHT OPTUNA RUN COMPLETE 🔥
Mapping optimal Optuna parameters directly to final models...

--- LightGBM ---
  learning_rate: 0.05789656828137544
  num_leaves: 80
  subsample: 0.8480595631138229
  colsample_bytree: 0.9173995865175877
  reg_lambda: 0.5726703217852122
  reg_alpha: 0.05336621674627308
  min_child_samples: 33
  min_split_gain: 0.002856610734086394
  max_bin: 147
  bagging_freq: 6

--- XGBoost ---
  learning_rate: 0.036231626460504394
  max_depth: 7
  subsample: 0.7788723052280968
  colsample_bytree: 0.9226315528614042
  reg_lambda: 0.10071174503562488
  reg_alpha: 0.03672239540543744
  min_child_weight: 3

--- CatBoost ---
  learning_rate: 0.14087436771639622
  depth: 7
  l2_leaf_reg: 9.975452672771397
  bagging_temperature: 0.02157765421122236
  random_strength: 3.9050992261570387

--- HistGBR ---
  learning_rate: 0.03583783709038413
  max_leaf_nodes: 115
  min_samples_leaf: 45
  l2_regularization: 0.07988585071126157
  max_bins: 231
---------------------------------

In [26]:
# ── Model definitions (Optuna Best Params + GPU + 3-Seed Averaging) ────────

lgb_params = {
    'objective': 'poisson',
    'n_estimators': 4000,
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': -1,
    **BEST_LGB_PARAMS,
}

xgb_params = {
    'objective': 'count:poisson',
    'eval_metric': 'rmse',
    'device': 'cuda',
    'tree_method': 'hist',
    'n_estimators': 4000,
    'random_state': SEED,
    'n_jobs': -1,
    'early_stopping_rounds': 150,
    **BEST_XGB_PARAMS,
}

cat_params = {
    'loss_function': 'Poisson',
    'task_type': 'GPU',
    'iterations': 4000,
    'random_seed': SEED,
    'verbose': 0,
    'allow_writing_files': False,
    **BEST_CAT_PARAMS,
}

hist_params = {
    'loss': 'poisson',
    'max_iter': 4000,
    'random_state': SEED,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 150,
    **BEST_HIST_PARAMS,
}


def train_lgb_fold(X_tr, y_tr, X_va, y_va, seed=SEED):
    params = {**lgb_params, 'random_state': seed}
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(150, verbose=False),
                   lgb.log_evaluation(-1)],
    )
    return model


def train_xgb_fold(X_tr, y_tr, X_va, y_va, seed=SEED):
    params = {**xgb_params, 'random_state': seed}
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )
    return model


def train_cat_fold(X_tr, y_tr, X_va, y_va, seed=SEED):
    params = {**cat_params, 'random_seed': seed}
    model = CatBoostRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        early_stopping_rounds=150,
        verbose=0,
    )
    return model


def train_hist_fold(X_tr, y_tr, X_va, y_va, seed=SEED):
    params = {**hist_params, 'random_state': seed}
    model = HistGradientBoostingRegressor(**params)
    model.fit(X_tr, y_tr)
    return model


def run_model_cv(model_name, X_train, y_target, X_test, all_folds, seed=SEED):
    """Single-seed CV for one model."""
    n_train = len(X_train)
    n_test  = len(X_test)
    oof_preds  = np.zeros(n_train)
    test_preds = np.zeros(n_test)
    fold_scores = []

    train_func = {
        "LightGBM": train_lgb_fold,
        "XGBoost":  train_xgb_fold,
        "CatBoost": train_cat_fold,
        "HistGBR":  train_hist_fold,
    }[model_name]

    for fold_idx, (tr_idx, va_idx) in enumerate(all_folds, 1):
        X_tr = X_train.iloc[tr_idx]
        y_tr = y_target.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_va = y_target.iloc[va_idx]

        model = train_func(X_tr, y_tr, X_va, y_va, seed=seed)

        va_pred   = model.predict(X_va)
        test_pred = model.predict(X_test)

        oof_preds[va_idx] = va_pred
        fold_r2 = r2_score(y_va, va_pred)
        fold_scores.append(fold_r2)

        test_preds += test_pred / N_SPLITS

    mean_r2 = float(np.mean(fold_scores))
    return oof_preds, test_preds, mean_r2


def run_model_cv_multiseed(model_name, X_train, y_target, X_test, all_folds,
                           seeds=None):
    """3-seed averaging: trains each model with multiple seeds, averages predictions."""
    if seeds is None:
        seeds = MULTI_SEEDS
    all_oof  = []
    all_test = []
    for i, seed in enumerate(seeds, 1):
        oof, test_p, r2 = run_model_cv(
            model_name, X_train, y_target, X_test, all_folds, seed=seed
        )
        all_oof.append(oof)
        all_test.append(test_p)
        print(f"  [{model_name}] seed {i}/{len(seeds)} (seed={seed}): CV R² = {r2:.5f}")
    avg_oof  = np.mean(all_oof,  axis=0)
    avg_test = np.mean(all_test, axis=0)
    avg_r2   = r2_score(y_target, avg_oof)
    print(f"  [{model_name}] 3-seed avg CV R² = {avg_r2:.5f}\n")
    return avg_oof, avg_test, avg_r2

print("Model functions defined (4 models, 3-seed averaging, GPU configs preserved)")

Model functions defined (4 models, 3-seed averaging, GPU configs preserved)


In [27]:
# ── SECTION 3: Train base models (3-seed averaging) ─────────────────────
print("Training base models (4 models × 3 seeds × 10 folds = 120 fits)...\n")

oof_lgb,  test_lgb,  r2_lgb  = run_model_cv_multiseed("LightGBM", X_train, y, X_test, all_folds)
oof_xgb,  test_xgb,  r2_xgb  = run_model_cv_multiseed("XGBoost",  X_train, y, X_test, all_folds)
oof_cat,  test_cat,  r2_cat  = run_model_cv_multiseed("CatBoost", X_train, y, X_test, all_folds)
oof_hist, test_hist, r2_hist = run_model_cv_multiseed("HistGBR",  X_train, y, X_test, all_folds)

Training base models (4 models × 3 seeds × 10 folds = 120 fits)...

  [LightGBM] seed 1/3 (seed=42): CV R² = 0.94262
  [LightGBM] seed 2/3 (seed=123): CV R² = 0.94268
  [LightGBM] seed 3/3 (seed=2024): CV R² = 0.94241
  [LightGBM] 3-seed avg CV R² = 0.94327

  [XGBoost] seed 1/3 (seed=42): CV R² = 0.93529
  [XGBoost] seed 2/3 (seed=123): CV R² = 0.93343
  [XGBoost] seed 3/3 (seed=2024): CV R² = 0.93158
  [XGBoost] 3-seed avg CV R² = 0.93305

  [CatBoost] seed 1/3 (seed=42): CV R² = 0.94032
  [CatBoost] seed 2/3 (seed=123): CV R² = 0.93959
  [CatBoost] seed 3/3 (seed=2024): CV R² = 0.94072
  [CatBoost] 3-seed avg CV R² = 0.94276

  [HistGBR] seed 1/3 (seed=42): CV R² = 0.94295
  [HistGBR] seed 2/3 (seed=123): CV R² = 0.94290
  [HistGBR] seed 3/3 (seed=2024): CV R² = 0.94306
  [HistGBR] 3-seed avg CV R² = 0.94398



In [28]:
from sklearn.linear_model import RidgeCV

S_train_oof = np.column_stack([oof_lgb, oof_xgb, oof_cat, oof_hist])
S_test_avg  = np.column_stack([test_lgb, test_xgb, test_cat, test_hist])

alphas_to_test = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
meta = RidgeCV(alphas=alphas_to_test, cv=5, fit_intercept=False)
meta.fit(S_train_oof, y)

print(f"  Optimal Alpha chosen: {meta.alpha_}")
print(f"  Ridge coefficients: LGB={meta.coef_[0]:.4f}, XGB={meta.coef_[1]:.4f}, "
      f"CAT={meta.coef_[2]:.4f}, Hist={meta.coef_[3]:.4f}")

final_oof  = meta.predict(S_train_oof)
final_pred = meta.predict(S_test_avg)

blend_cv_r2 = r2_score(y, final_oof)
print(f"  Ridge OOF R²  = {blend_cv_r2:.5f}")
print(f"  Estimated score = {max(0, 100*blend_cv_r2):.4f}")

  Optimal Alpha chosen: 1.0
  Ridge coefficients: LGB=0.1705, XGB=0.0395, CAT=0.3516, Hist=0.4467
  Ridge OOF R²  = 0.94489
  Estimated score = 94.4886


In [31]:
# ── SECTION 5: Post-processing + Isotonic Calibration ───────────────────

# 1. Isotonic Calibration: match training target distribution
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(final_oof, y)
final_pred_calibrated = iso.predict(final_pred)
print("  Isotonic calibration applied")

# 2. Clip to [0, p99.5]
p995 = np.percentile(train_raw[TARGET].dropna().values, 99.5)
final_pred_calibrated = np.clip(final_pred_calibrated, 0, p995)
print(f"  Clipped to [0, {p995:.4f}]")

# 3. Rounding (only if target values are 0.5-step aligned)
vals = train_raw[TARGET].dropna().values
if np.mean(vals % 0.5 == 0) > 0.95:
    final_pred_calibrated = np.round(final_pred_calibrated * 2) / 2
    print("  Applied 0.5-step rounding")
else:
    print("  No rounding applied")

# 4. Save
submission = pd.DataFrame({ID_COL: test_raw[ID_COL], TARGET: final_pred_calibrated})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"  Saved {SUBMISSION_PATH}  shape={submission.shape}")

  Isotonic calibration applied
  Clipped to [0, 1.0000]
  No rounding applied
  Saved submission.csv  shape=(41778, 2)


In [32]:
# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("═" * 50)
print(f"  LightGBM  CV R² : {r2_lgb:.5f}")
print(f"  XGBoost   CV R² : {r2_xgb:.5f}")
print(f"  CatBoost  CV R² : {r2_cat:.5f}")
print(f"  HistGBR   CV R² : {r2_hist:.5f}")
print(f"  Ridge     OOF R²: {blend_cv_r2:.5f}")
print(f"  Estimated score : {max(0, 100*blend_cv_r2):.4f}")
print("═" * 50)


══════════════════════════════════════════════════
  LightGBM  CV R² : 0.94327
  XGBoost   CV R² : 0.93305
  CatBoost  CV R² : 0.94276
  HistGBR   CV R² : 0.94398
  Ridge     OOF R²: 0.94489
  Estimated score : 94.4886
══════════════════════════════════════════════════
